To run this notebook, additionally install the following modules:

`pip install ipykernel ipympl`

In [1]:
%load_ext autoreload
%autoreload 2

%matplotlib widget

In [ ]:
import os
import torch
from PIL import Image
import matplotlib.pyplot as plt
from utils.utils_correspondence import resize
from model_utils.extractor_dino import ViTExtractor
from model_utils.projection_network import AggregationNetwork
from preprocess_map import set_seed

set_seed(42)
num_patches = 60
aggre_net = AggregationNetwork(feature_dims=[768,], projection_dim=768, device='cuda')
ckpt_file = "../ckpts/0300_dino_spair/best.pth"
aggre_net.load_pretrained_weights(torch.load(ckpt_file))

def get_processed_features_dino(aggre_net, extractor_vit, num_patches, img=None, img_path=None):
    
    if img_path is not None:
        feature_base = img_path.replace('JPEGImages', 'features').replace('.jpg', '')
        dino_path = f"{feature_base}_dino.pt"

    if img_path is not None and os.path.exists(dino_path):
        features_dino = torch.load(dino_path)
    else:
        if img is None: img = Image.open(img_path).convert('RGB')
        img_dino_input = resize(img, target_res=num_patches*14, resize=True, to_pil=True)
        img_batch = extractor_vit.preprocess_pil(img_dino_input)
        features_dino = extractor_vit.extract_descriptors(img_batch.cuda(), layer=11, facet='token').permute(0, 1, 3, 2).reshape(1, -1, num_patches, num_patches)

    desc_gathered = features_dino

    with torch.no_grad():
        desc = aggre_net(desc_gathered)
    norms_desc = torch.linalg.norm(desc, dim=1, keepdim=True)
    desc = desc / (norms_desc + 1e-8)
    return desc

In [ ]:
extractor_vit = ViTExtractor('dinov2_vitb14', stride=14, device='cuda')

In [ ]:
img1_path = '../assets/bus_1.JPEG'
img2_path = '../assets/bus_2.JPEG'


img_size = 240
img1 = resize(Image.open(img1_path).convert('RGB'), target_res=img_size, resize=True, to_pil=True)
img2 = resize(Image.open(img2_path).convert('RGB'), target_res=img_size, resize=True, to_pil=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 5))
for a in ax: a.axis('off')
ax[0].imshow(img1)
ax[0].set_title('source image')
ax[1].imshow(img2)
ax[1].set_title('target image')
plt.show()

feat1 = get_processed_features_dino(aggre_net, extractor_vit, num_patches, img=img1)
feat2 = get_processed_features_dino(aggre_net, extractor_vit, num_patches, img=img2)


In [ ]:
%matplotlib widget
from utils.utils_visualization_demo import Demo

demo = Demo([img1,img2], torch.cat([feat1, feat2], dim=0), img_size)
demo.plot_img_pairs(fig_size=5)